### RAG PIPILANES - DATA INGESTION TO VECTOR DB PIPELAIN

In [1]:
import os 
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from pathlib import Path

c:\Users\HP\Documents\works\RAG test\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def process_all_pdfs(pdf_directory):
    """ process all PDF files in the directory"""
    all_documents= []
    pdf_dir = Path (pdf_directory)
    
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"found{len(pdf_files)} PDF file process")

    for pdf_file in pdf_files:
        print(f"\nprocessing:{pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = 'pdf'
            all_documents.extend(documents)
            print(f"loaded{len(documents)} pages")
        except Exception as e:
            print(f'ERROR:{e}')
    print(f"\ntotal documents loaded:{len(all_documents)}")
    return all_documents


all_pdf_documents = process_all_pdfs("../data")

found2 PDF file process

processing:انسان در جستجوی معنا - ویکتور فرانکل.pdf
loaded48 pages

processing:اوضاع خیلی خراب است مارک منسن  (1).Pdf
loaded148 pages

total documents loaded:196


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'pdfFactory Pro 4.0 (Windows XP Professional)', 'creator': 'pdfFactory Pro www.pdffactory.com', 'creationdate': '2011-09-12T13:31:08+04:30', 'title': 'ensan dar jostojuoye mana', 'author': 'heydari', 'source': '..\\data\\pdf\\انسان در جستجوی معنا - ویکتور فرانکل.pdf', 'total_pages': 48, 'page': 0, 'page_label': '1', 'source_file': 'انسان در جستجوی معنا - ویکتور فرانکل.pdf', 'file_type': 'pdf'}, page_content='١ \naspx.151-post/com.blogfa.aarmaan://http  \n  \nا ﻣﻌﻨﯽ ﺟﺴﺘﺠﻮي در ﻧﺴﺎن \n ﻧﻮﺷﺘﻪ : ﻓﺮاﻧﮑﻞ وﯾﮑﺘﻮر دﮐﺘﺮ  \n وﯾﻦ داﻧﺸﮕﺎه ﭘﺰﺷﮑﯽ روان اﺳﺘﺎد  \n ﺗﺮﺟﻤﻪ : ﻣﻌﺎرﻓﯽ اﮐﺒﺮ دﮐﺘﺮ  \n  \n \nﯿﺗﻮﺿ ﻓﺎرﺳﯽ ﻣﺘﺮﺟﻢ  ﺤﺎت \nﮔﻮردو دﮐﺘﺮ ﮐﻪ درآﻣﺪي ﭘﯿﺶ ﺑﺎ ﻣﻘﺪﻣﻪ ﻧﺪارد ﺟﺎ اﺳﺖ ﻧﻮﺷﺘﻪ ﮐﺘﺎب اﯾﻦ ﺑﺮ ﻫﺎروارد داﻧﺸﮕﺎه رواﻧﺸﻨﺎﺳﯽ ﭘﯿﺸﯿﻦ اﺳﺘﺎد آﻟﭙﻮرت ن\nﻣ را ﺗﻮﺿﯿﺤﺎﺗﯽ ﺗﻨﻬﺎ ﻣﻦ و ﺷﻮد ﻧﻮﺷﺘﻪ دﯾﮕﺮي ﯽدراز  ﻧﯿﺴﺖ ﻧﻮﺷﺘﻪ آن در ﮐﻪ اﻓﺰاﯾﻢ  . \n ﯾﻌﻨﯽ ﮐﺘﺎب اﯾﻦ اول اﺳﯿﺮان اردوي در ﺳﺮﮔﺸﺘﯽ «   را آن ﻣـﻦ و اﺳـﺖ ﻧﻮﺷﺘﻪ آﻟﻤﺎﻧﯽ ﺑﺰﺑﺎن ﻓﺮاﻧﮑﻞ دﮐﺘﺮ را  ﺗﺮﺟﻤـﻪ از\nام درآورده ﺑﻔﺎرﺳﯽ ﻻش اﯾﻠﺰه اﻧﮕﻠﯿﺴﯽ . ﯾﻌﻨﯽ دﯾﮕﺮ ﺑﺨﺶ ﻟﻮﮔﻮﺗ

In [4]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """ split documents into smaller chunks for better RAG performance"""
    text_splitter= RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators=["\n\n","\n"," ",""]
    )
    split_doc = text_splitter.split_documents(documents)
    print(f"split{len(documents)} documents into {len(split_doc)}chunks")

    if split_doc:
        print(f"\nexample:")
        print(f"content:{split_doc[0].page_content[:200]}....")
        print(f"metadata:{split_doc[0].metadata}")
    return split_doc

In [5]:
chunks = split_documents(all_pdf_documents)
chunks

split196 documents into 384chunks

example:
content:١ 
aspx.151-post/com.blogfa.aarmaan://http  
  
ا ﻣﻌﻨﯽ ﺟﺴﺘﺠﻮي در ﻧﺴﺎن 
 ﻧﻮﺷﺘﻪ : ﻓﺮاﻧﮑﻞ وﯾﮑﺘﻮر دﮐﺘﺮ  
 وﯾﻦ داﻧﺸﮕﺎه ﭘﺰﺷﮑﯽ روان اﺳﺘﺎد  
 ﺗﺮﺟﻤﻪ : ﻣﻌﺎرﻓﯽ اﮐﺒﺮ دﮐﺘﺮ  
  
 
ﯿﺗﻮﺿ ﻓﺎرﺳﯽ ﻣﺘﺮﺟﻢ  ﺤﺎت 
ﮔﻮردو دﮐﺘﺮ ....
metadata:{'producer': 'pdfFactory Pro 4.0 (Windows XP Professional)', 'creator': 'pdfFactory Pro www.pdffactory.com', 'creationdate': '2011-09-12T13:31:08+04:30', 'title': 'ensan dar jostojuoye mana', 'author': 'heydari', 'source': '..\\data\\pdf\\انسان در جستجوی معنا - ویکتور فرانکل.pdf', 'total_pages': 48, 'page': 0, 'page_label': '1', 'source_file': 'انسان در جستجوی معنا - ویکتور فرانکل.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'pdfFactory Pro 4.0 (Windows XP Professional)', 'creator': 'pdfFactory Pro www.pdffactory.com', 'creationdate': '2011-09-12T13:31:08+04:30', 'title': 'ensan dar jostojuoye mana', 'author': 'heydari', 'source': '..\\data\\pdf\\انسان در جستجوی معنا - ویکتور فرانکل.pdf', 'total_pages': 48, 'page': 0, 'page_label': '1', 'source_file': 'انسان در جستجوی معنا - ویکتور فرانکل.pdf', 'file_type': 'pdf'}, page_content='١ \naspx.151-post/com.blogfa.aarmaan://http  \n  \nا ﻣﻌﻨﯽ ﺟﺴﺘﺠﻮي در ﻧﺴﺎن \n ﻧﻮﺷﺘﻪ : ﻓﺮاﻧﮑﻞ وﯾﮑﺘﻮر دﮐﺘﺮ  \n وﯾﻦ داﻧﺸﮕﺎه ﭘﺰﺷﮑﯽ روان اﺳﺘﺎد  \n ﺗﺮﺟﻤﻪ : ﻣﻌﺎرﻓﯽ اﮐﺒﺮ دﮐﺘﺮ  \n  \n \nﯿﺗﻮﺿ ﻓﺎرﺳﯽ ﻣﺘﺮﺟﻢ  ﺤﺎت \nﮔﻮردو دﮐﺘﺮ ﮐﻪ درآﻣﺪي ﭘﯿﺶ ﺑﺎ ﻣﻘﺪﻣﻪ ﻧﺪارد ﺟﺎ اﺳﺖ ﻧﻮﺷﺘﻪ ﮐﺘﺎب اﯾﻦ ﺑﺮ ﻫﺎروارد داﻧﺸﮕﺎه رواﻧﺸﻨﺎﺳﯽ ﭘﯿﺸﯿﻦ اﺳﺘﺎد آﻟﭙﻮرت ن\nﻣ را ﺗﻮﺿﯿﺤﺎﺗﯽ ﺗﻨﻬﺎ ﻣﻦ و ﺷﻮد ﻧﻮﺷﺘﻪ دﯾﮕﺮي ﯽدراز  ﻧﯿﺴﺖ ﻧﻮﺷﺘﻪ آن در ﮐﻪ اﻓﺰاﯾﻢ  . \n ﯾﻌﻨﯽ ﮐﺘﺎب اﯾﻦ اول اﺳﯿﺮان اردوي در ﺳﺮﮔﺸﺘﯽ «   را آن ﻣـﻦ و اﺳـﺖ ﻧﻮﺷﺘﻪ آﻟﻤﺎﻧﯽ ﺑﺰﺑﺎن ﻓﺮاﻧﮑﻞ دﮐﺘﺮ را  ﺗﺮﺟﻤـﻪ از\nام درآورده ﺑﻔﺎرﺳﯽ ﻻش اﯾﻠﺰه اﻧﮕﻠﯿﺴﯽ . ﯾﻌﻨﯽ دﯾﮕﺮ ﺑﺨﺶ ﻟﻮﮔﻮﺗ

### EMBEDING AND VECTORSTOREDB

In [6]:

import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb 
from chromadb.config import Settings
import uuid
from typing import List , Dict , Any ,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:
    """handles documents embedding generation using SentenceTransformer"""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        initialize the embedding manager

        args:
            model_name: huggingface model name for sentence embedding
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """load the SentenceTransformer model"""
        try:
            print(f"loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"model loaded successfully, embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"error loading model {self.model_name}: {e}")
            raise

    def generate_embedding(self, texts: list[str]) -> np.ndarray:
        """
        generate embedding for a list of text 

        args:
            texts: list of text string to embed
        return:
            numpy array of embeddings with shape (len(texts), embeddings_dim)
        """
        if not self.model:
            raise ValueError("model not loaded")

        print(f"generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"generated embeddings with shape: {embeddings.shape}")
        return embeddings

    def get_embedding_dimension(self) -> int:
        """get the embedding dimension of the model"""
        if not self.model:
            raise ValueError("model not loaded")
        return self.model.get_sentence_embedding_dimension()

# استفاده
embedding_manager = EmbeddingManager()

loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1471.70it/s]


model loaded successfully, embedding dimension: 384


C:\Users\HP\AppData\Local\Temp\ipykernel_10032\3034071468.py:19: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"model loaded successfully, embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### VECTOR STORE 

In [13]:

class VectorStore:
    """manage document embeddings in a chromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """initialize the chromaDB client and collection"""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "pdf document embedding for RAG"}
            )
            print(f"vector store initialized with collection name {self.collection_name}")
            print(f"existing document in collection : {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing the vector store: {e}")
            raise

    # دقت کن: این متد باید هم‌سطح با _initialize_store باشه، نه داخلش
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """add document to and their embedding to the vector store"""
        if len(documents) != len(embeddings):
            raise ValueError("the number of documents and embeddings must be equal")

        print(f"adding {len(documents)} documents to vector store")

        ids = []
        metadatas = []
        documents_text = []
        embedding_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # کپی متادیتا + پاکسازی مقادیر None که کروما قبول نمیکنه
            meta = dict(doc.metadata) if hasattr(doc, "metadata") else {}
            meta = {k: v for k, v in meta.items() if v is not None}
            # تبدیل مقادیر غیر ساده به str
            for k, v in list(meta.items()):
                if not isinstance(v, (str, int, float, bool)):
                    meta[k] = str(v)
            meta["doc_index"] = i
            meta["content_length"] = len(doc.page_content)
            metadatas.append(meta)

            documents_text.append(doc.page_content)
            # اگر numpy هست به list تبدیل کن، اگر list هست همون بمونه
            if hasattr(embedding, "tolist"):
                embedding_list.append(embedding.tolist())
            else:
                embedding_list.append(list(embedding))

        try:
            self.collection.add(
                ids=ids,
                embeddings=embedding_list,  # جمع، نه مفرد
                metadatas=metadatas,        # جمع، نه مفرد
                documents=documents_text
            )
            print(f"successfully added {len(documents)} documents to vector store")
            print(f"total documents in collection : {self.collection.count()}")
        except Exception as e:
            print(f"error adding doc to vector store {e}")
            raise

In [14]:
chunks

[Document(metadata={'producer': 'pdfFactory Pro 4.0 (Windows XP Professional)', 'creator': 'pdfFactory Pro www.pdffactory.com', 'creationdate': '2011-09-12T13:31:08+04:30', 'title': 'ensan dar jostojuoye mana', 'author': 'heydari', 'source': '..\\data\\pdf\\انسان در جستجوی معنا - ویکتور فرانکل.pdf', 'total_pages': 48, 'page': 0, 'page_label': '1', 'source_file': 'انسان در جستجوی معنا - ویکتور فرانکل.pdf', 'file_type': 'pdf'}, page_content='١ \naspx.151-post/com.blogfa.aarmaan://http  \n  \nا ﻣﻌﻨﯽ ﺟﺴﺘﺠﻮي در ﻧﺴﺎن \n ﻧﻮﺷﺘﻪ : ﻓﺮاﻧﮑﻞ وﯾﮑﺘﻮر دﮐﺘﺮ  \n وﯾﻦ داﻧﺸﮕﺎه ﭘﺰﺷﮑﯽ روان اﺳﺘﺎد  \n ﺗﺮﺟﻤﻪ : ﻣﻌﺎرﻓﯽ اﮐﺒﺮ دﮐﺘﺮ  \n  \n \nﯿﺗﻮﺿ ﻓﺎرﺳﯽ ﻣﺘﺮﺟﻢ  ﺤﺎت \nﮔﻮردو دﮐﺘﺮ ﮐﻪ درآﻣﺪي ﭘﯿﺶ ﺑﺎ ﻣﻘﺪﻣﻪ ﻧﺪارد ﺟﺎ اﺳﺖ ﻧﻮﺷﺘﻪ ﮐﺘﺎب اﯾﻦ ﺑﺮ ﻫﺎروارد داﻧﺸﮕﺎه رواﻧﺸﻨﺎﺳﯽ ﭘﯿﺸﯿﻦ اﺳﺘﺎد آﻟﭙﻮرت ن\nﻣ را ﺗﻮﺿﯿﺤﺎﺗﯽ ﺗﻨﻬﺎ ﻣﻦ و ﺷﻮد ﻧﻮﺷﺘﻪ دﯾﮕﺮي ﯽدراز  ﻧﯿﺴﺖ ﻧﻮﺷﺘﻪ آن در ﮐﻪ اﻓﺰاﯾﻢ  . \n ﯾﻌﻨﯽ ﮐﺘﺎب اﯾﻦ اول اﺳﯿﺮان اردوي در ﺳﺮﮔﺸﺘﯽ «   را آن ﻣـﻦ و اﺳـﺖ ﻧﻮﺷﺘﻪ آﻟﻤﺎﻧﯽ ﺑﺰﺑﺎن ﻓﺮاﻧﮑﻞ دﮐﺘﺮ را  ﺗﺮﺟﻤـﻪ از\nام درآورده ﺑﻔﺎرﺳﯽ ﻻش اﯾﻠﺰه اﻧﮕﻠﯿﺴﯽ . ﯾﻌﻨﯽ دﯾﮕﺮ ﺑﺨﺶ ﻟﻮﮔﻮﺗ

In [15]:
texts = [doc.page_content for doc in chunks]
texts

['١ \naspx.151-post/com.blogfa.aarmaan://http  \n  \nا ﻣﻌﻨﯽ ﺟﺴﺘﺠﻮي در ﻧﺴﺎن \n ﻧﻮﺷﺘﻪ : ﻓﺮاﻧﮑﻞ وﯾﮑﺘﻮر دﮐﺘﺮ  \n وﯾﻦ داﻧﺸﮕﺎه ﭘﺰﺷﮑﯽ روان اﺳﺘﺎد  \n ﺗﺮﺟﻤﻪ : ﻣﻌﺎرﻓﯽ اﮐﺒﺮ دﮐﺘﺮ  \n  \n \nﯿﺗﻮﺿ ﻓﺎرﺳﯽ ﻣﺘﺮﺟﻢ  ﺤﺎت \nﮔﻮردو دﮐﺘﺮ ﮐﻪ درآﻣﺪي ﭘﯿﺶ ﺑﺎ ﻣﻘﺪﻣﻪ ﻧﺪارد ﺟﺎ اﺳﺖ ﻧﻮﺷﺘﻪ ﮐﺘﺎب اﯾﻦ ﺑﺮ ﻫﺎروارد داﻧﺸﮕﺎه رواﻧﺸﻨﺎﺳﯽ ﭘﯿﺸﯿﻦ اﺳﺘﺎد آﻟﭙﻮرت ن\nﻣ را ﺗﻮﺿﯿﺤﺎﺗﯽ ﺗﻨﻬﺎ ﻣﻦ و ﺷﻮد ﻧﻮﺷﺘﻪ دﯾﮕﺮي ﯽدراز  ﻧﯿﺴﺖ ﻧﻮﺷﺘﻪ آن در ﮐﻪ اﻓﺰاﯾﻢ  . \n ﯾﻌﻨﯽ ﮐﺘﺎب اﯾﻦ اول اﺳﯿﺮان اردوي در ﺳﺮﮔﺸﺘﯽ «   را آن ﻣـﻦ و اﺳـﺖ ﻧﻮﺷﺘﻪ آﻟﻤﺎﻧﯽ ﺑﺰﺑﺎن ﻓﺮاﻧﮑﻞ دﮐﺘﺮ را  ﺗﺮﺟﻤـﻪ از\nام درآورده ﺑﻔﺎرﺳﯽ ﻻش اﯾﻠﺰه اﻧﮕﻠﯿﺴﯽ . ﯾﻌﻨﯽ دﯾﮕﺮ ﺑﺨﺶ ﻟﻮﮔﻮﺗﺮاﭘﯽ اﺳﺎﺳﯽ ﻣﻔﻬﻮم اﺳﺖ ﻧﻮﺷﺘﻪ ﺑﺎﻧﮕﻠﯿﺴﯽ ﻓﺮاﻧﮑﻞ را  . \n در ﮐﻪ ﮐﺘﺎب اﯾﻦ1963  ﻋﻨﻮان ﺑﺎMan’s search for Meaning           ﺑـﺰودي ﮐـﻪ ﯾﺎﻓـﺖ ﺷـﻬﺮﺗﯽ ﭼﻨـﺎن ﺷـﺪ ﻣﻨﺘﺸـﺮ\nرﺳﯿﺪ ﭼﺎپ ﺑﭽﻨﺪﯾﻦ. ﻧﺴﺨﻪ ﮐﺮدم ﺗﺮﺟﻤﻪ آن از ﻣﻦ ﮐﻪ اي  ﺳﺎل در و اﺳﺖ ﯾﺎزدﻫﻢ ﭼﺎپ1967 ﺷﺪه ﻣﻨﺘﺸﺮ  اﺳﺖ  . \n ﺑﺴﺎل ﻓﺮاﻧﮑﻞ دﮐﺘﺮ1905  در ﺷﻬﺮ ﻫﻤﺎن در و آﻣﺪ ﺑﺪﻧﯿﺎ وﯾﻦ در1930     در و رﺳـﺎﻧﯿﺪ ﺑﭙﺎﯾـﺎن را ﭘﺰﺷـﮑﯽ داﻧﺸﮑﺪه1942   ﮐـﻪ',
 'ﺑﺴﺎل ﻓﺮاﻧﮑﻞ دﮐﺘﺮ1905  در ﺷﻬﺮ ﻫﻤﺎن در و آﻣﺪ ﺑﺪﻧﯿﺎ وﯾﻦ در1930     در و ر

In [17]:
# اسم متغیر رو با اسم کلاس یکی نذار تا کلاس shadow نشه
vector_store = VectorStore()
vector_store

vector store initialized with collection name pdf_documents
existing document in collection : 0


In [18]:
texts = [doc.page_content for doc in chunks]
embeddings = embedding_manager.generate_embedding(texts)
vector_store.add_documents(chunks, embeddings)

generating embeddings for 384 texts...


Batches: 100%|██████████| 12/12 [00:10<00:00,  1.11it/s]


generated embeddings with shape: (384, 384)
adding 384 documents to vector store
successfully added 384 documents to vector store
total documents in collection : 384


### RETRIEVER PIPELINE FROM VECTORTOR STORE

In [26]:
class RAGRetriever:
    def __init__(self , vector_store: vectorstore , embedding_manager: EmbeddingManager):
        """
        initialize retriever
        Args:
            vector_store : VectorStore containing documents and embeddings
            embedding_manager : EmbeddingManager for generating embeddings
        """
        self.vector_store = vector_store 
        self.embedding_manager = embedding_manager
def retrieve(self , query: str , top_k : int = 5 , score_threshold: float = 0.0) -> List[dict[str,Any]]:
    """
    retriever relevant documents for a query
    Args:
        query : search query
        top_k : number of top results to return
        score_threshold: minimum similarity score threshold  
        returns:
            list of dictionaries containing document metadata and scores
    """
    print(f"retrieving documents for {query}")
    print(f"top k : {top_k} , score threshold : {score_threshold}")

    query_embedding = self.embedding_manager.get_embeddings([query])[0]

    try:
        results = self.vector_store.collection.query(
            query_embedding = [query_embedding.tolist()],
            n_result=top_k
        )
        retrieved_docs = []

        if results['documents'] and results['documents'][0]:
            documents = results["documents"][0]
            metadata = results["metadata"][0]
            distances = results["distances"][0]
            ids = results["ids"][0]

            for i,(doc_id , documents , metadata ,distance) in enumerate(zip(doc_id , documents , metadata ,distances)):
                similarity_score = 1 - distance 

            if similarity_score >= score_threshold:
                retrieved_docs.append({
                    "document":documents,
                    "metadata":metadata , 
                    "content":documents, 
                    "id":doc_id , 
                    "similarity_score": similarity_score , 
                    "distance": distance , 
                    "rank": i + 1
                })
            print(f"retrieved {len(retrieved_docs)} documents from the index")
        else:
           print("no documents found ")
        return retrieved_docs
    except Exception as e :
        print(f"error during retrieval: {e}")
        return []
    rag_retriever = RAGRetriever(vectorstore , embedding_manager)

In [27]:
rag_retriever

NameError: name 'rag_retriever' is not defined